# The Souled Store — End-to-End Data Engineering Project

## Project Overview
This project extracts product data from The Souled Store API
and processes it using Python, Databricks, PySpark, Delta Lake,
and SQL.

## Architecture
API → Python → CSV → Databricks → Bronze → Silver → Gold

## Technologies Used
- Python
- REST API
- Azure Databricks Free Edition
- PySpark
- SQL
- Delta Lake
- Medallion Architecture

## Data Source
The Souled Store — Men's Footwear Products

## Project Objectives
1. Extract product data using Python.
2. Store the extracted data as a CSV file.
3. Load the CSV into Databricks.
4. Create a Bronze Delta table.
5. Clean and validate data in the Silver layer.
6. Generate business summaries in the Gold layer.
7. Analyse products using SQL.

## Final Output
A category-level product summary containing product counts,
average prices, highest prices, and total stock.

In [0]:
# ==================================================
# THE SOULED STORE - DATA ENGINEERING PROJECT
# Step 1: Read the uploaded CSV
# ==================================================

from pyspark.sql import functions as F

file_path = "/Volumes/workspace/default/soulstore_volume/soul_store_products.csv"

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(file_path)
)

print("Source data loaded successfully!")
print("Total products:", df.count())

display(df)

Source data loaded successfully!
Total products: 24


id,title,price,category,product_slug,stock,artist
331297,Urban Blaze: Terra,3499,Men Low Top Sneakers,souled-green-ghost-low-top-sneakers-men,12,The Souled Store
364368,Yugen: Brown,2599,Men Low Top Sneakers,casuals-sienna-men-low-top-sneakers,12,The Souled Store
364376,Yugen: Black,2599,Men Low Top Sneakers,casuals-sable-men-low-top-sneakers,12,The Souled Store
364392,Yugen: Green,2599,Men Low Top Sneakers,casuals-forest-men-low-top-sneakers,12,The Souled Store
317498,Flintstones: Stone Age,2599,Men Low Top Sneakers,flintstones-stone-age-men-low-top-sneakers,12,The Flintstones™
336528,Batman: Vengeance,2599,Men Low Top Sneakers,batman-urban-knight-men-low-top-sneakers,12,DC Comics™
348978,UBZ: Black Panther,2599,Men Low Top Sneakers,black-panther-rega-men-low-top-sneakers,12,Marvel™
367080,Spiderman: Slides,1299,Men Sliders,spider-man-red-web-men-slider,12,Marvel™
366799,Spider-Man: Brand New Day,3999,Men Mid Top Sneakers,spider-man-crawler-men-midtop-sneaker,12,Marvel™
310047,UBZ NXT: Off White,2599,Men Low Top Sneakers,ubz-triple-2-men-low-top-sneakers,12,The Souled Store


In [0]:
# BRONZE LAYER: Store raw source data in Delta format

bronze_table = "workspace.default.soulstore_bronze"

df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(bronze_table)

print("Bronze layer completed!")
print("Bronze row count:", spark.table(bronze_table).count())

Bronze layer completed!
Bronze row count: 24


In [0]:
# SILVER LAYER: Clean and validate product data

bronze_df = spark.table("workspace.default.soulstore_bronze")

# Remove duplicate IDs and invalid essential fields
silver_df = (
    bronze_df
    .dropDuplicates(["id"])
    .filter(F.col("id").isNotNull())
    .filter(F.col("title").isNotNull())
    .filter(F.col("price").isNotNull())
    .filter(F.col("price") >= 0)
)

# Save the cleaned data
silver_table = "workspace.default.soulstore_silver"

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(silver_table)

print("Silver layer completed!")
print("Bronze row count:", bronze_df.count())
print("Silver row count:", silver_df.count())
print("Removed rows:", bronze_df.count() - silver_df.count())

Silver layer completed!
Bronze row count: 24
Silver row count: 24
Removed rows: 0


In [0]:
# GOLD LAYER: Create business analytics summary

silver_table = "workspace.default.soulstore_silver"
gold_table = "workspace.default.soulstore_gold"

silver_df = spark.table(silver_table)

gold_df = (
    silver_df
    .groupBy("category")
    .agg(
        F.count("*").alias("total_products"),
        F.round(F.avg("price"), 2).alias("average_price"),
        F.max("price").alias("highest_price"),
        F.sum("stock").alias("total_stock")
    )
    .orderBy(F.desc("total_products"))
)

gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(gold_table)

print("Gold layer completed!")
print("Gold summary rows:", spark.table(gold_table).count())

display(spark.table(gold_table))

Gold layer completed!
Gold summary rows: 5


category,total_products,average_price,highest_price,total_stock
Men Low Top Sneakers,15,2739.0,3799,180
Men Sliders,5,1299.0,1299,60
Men High Top Sneakers,2,4049.0,4499,24
Men Mules,1,3499.0,3499,2
Men Mid Top Sneakers,1,3999.0,3999,12


In [0]:
# FINAL SQL ANALYSIS: Top 5 most expensive products

top_products = spark.sql("""
    SELECT
        id,
        title,
        category,
        price,
        stock
    FROM workspace.default.soulstore_silver
    ORDER BY price DESC
    LIMIT 5
""")

display(top_products)

id,title,category,price,stock
324402,Batman: Gotham Blaze,Men High Top Sneakers,4499,12
366799,Spider-Man: Brand New Day,Men Mid Top Sneakers,3999,12
328902,Raptors V2: Canyon Dust,Men Low Top Sneakers,3799,12
310093,Deadpool: Maximum Effort,Men High Top Sneakers,3599,12
269980,Rovers: Batman,Men Mules,3499,2


# The Souled Store — Data Engineering Project

## Project Overview
An end-to-end data engineering pipeline built using Python,
Databricks, PySpark, SQL, and Delta Lake.

## Data Source
The Souled Store men's footwear product listing API.

## Technologies Used
- Python and Requests — API data extraction
- CSV — Raw data landing format
- Databricks — Cloud data processing
- PySpark — Data transformation and cleaning
- Delta Lake — Storage for Bronze, Silver, and Gold layers
- SQL — Business analysis

## Medallion Architecture

### Bronze Layer
Stores the source product data in Delta format.

### Silver Layer
Cleans and validates product records and checks duplicate IDs.

### Gold Layer
Creates category-level business summaries:
- Total products
- Average price
- Highest price
- Total stock

## SQL Analysis
Identifies the five most expensive products.

## Results
- 24 products extracted
- 24 Bronze records
- 24 Silver records
- 5 category-level Gold summary rows

## Conclusion
Built a working data pipeline from API extraction
to cloud data processing and business analysis.